In [46]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

**Q.1 Check version of Spark install on machine**

In [5]:
spark.version

'4.1.1'

**Q.2 What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.**

In [6]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-01 19:58:27--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.85.97.194, 52.85.97.61, 52.85.97.44, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.97.194|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  26.8MB/s    in 2.5s    

2026-03-01 19:58:30 (26.8 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [24]:
df = spark.read \
    .option("header", "true") \
    .schema()
    .parquet('yellow_tripdata_2025-11.parquet')

In [65]:
yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.LongType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.LongType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.LongType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [66]:
df = spark.read \
    .option("header", "true") \
    .schema(yellow_schema) \
    .parquet('yellow_tripdata_2025-11.parquet')

In [67]:
df = df.repartition(4)

In [68]:
df.write.parquet('yellow/2025/11/',mode='overwrite')

In [69]:
!cd yellow/2025/11 && ls -lh

total 104M
-rw-r--r-- 1 NEOPOSTADVPatel_na NEOPOSTADVPatel_na   0 Mar  1 21:35 _SUCCESS
-rw-r--r-- 1 NEOPOSTADVPatel_na NEOPOSTADVPatel_na 26M Mar  1 21:35 part-00000-a070efc9-8046-4d4c-83ff-15bf6f1b72ce-c000.snappy.parquet
-rw-r--r-- 1 NEOPOSTADVPatel_na NEOPOSTADVPatel_na 26M Mar  1 21:35 part-00001-a070efc9-8046-4d4c-83ff-15bf6f1b72ce-c000.snappy.parquet
-rw-r--r-- 1 NEOPOSTADVPatel_na NEOPOSTADVPatel_na 27M Mar  1 21:35 part-00002-a070efc9-8046-4d4c-83ff-15bf6f1b72ce-c000.snappy.parquet
-rw-r--r-- 1 NEOPOSTADVPatel_na NEOPOSTADVPatel_na 26M Mar  1 21:35 part-00003-a070efc9-8046-4d4c-83ff-15bf6f1b72ce-c000.snappy.parquet


**Q.3 How many taxi trips were there on the 15th of November?**

In [70]:
df = spark.read.parquet('yellow/2025/11/')

In [71]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [73]:
from pyspark.sql import functions as F

In [72]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       2| 2025-11-01 21:30:47|  2025-11-01 21:43:13|              1|         2.99|         1|                 N|          87|           4|           1|       16.3|  1.0|    0.5|       1.0|         0.0|                  1.0

In [75]:
df.createOrReplaceTempView('yellownov2025')

In [76]:
spark.sql("""
SELECT    
    count(1)
FROM
    yellownov2025
WHERE tpep_pickup_datetime >= '2025-11-15 00:00:00' and tpep_pickup_datetime <= '2025-11-15 23:59:59'
""").show()

[Stage 27:>                                                         (0 + 4) / 4]

+--------+
|count(1)|
+--------+
|  162604|
+--------+



**Q.4 What is the length of the longest trip in the dataset in hours?**

In [79]:
spark.sql("""
SELECT    
    max(TIMESTAMPDIFF(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime) ) as max_trip_duration
FROM
    yellownov2025
""").show()

+-----------------+
|max_trip_duration|
+-----------------+
|               90|
+-----------------+



**Q.6 Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?**

In [82]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-01 22:17:53--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.85.97.194, 52.85.97.44, 52.85.97.61, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.97.194|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-01 22:17:53 (151 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [86]:
df_zones = spark.read \
            .option("header", "true") \
            .csv('taxi_zone_lookup.csv')

In [87]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [89]:
df_result = df.join(df_zones, df.PULocationID == df_zones.LocationID)

In [91]:
df_result.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+----------+---------+--------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|LocationID|  Borough|                Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+----------+---------+--------------------+------------+
|       2| 2025-11-01 21:30:47|  2025-11-01 21:43:13|     

In [90]:
df_result.createOrReplaceTempView('leastpickuploc')

In [93]:
spark.sql("""
SELECT    
    Zone,
    count(1)
FROM
    leastpickuploc
group by
    Zone
order by 2
""").show()

[Stage 50:==============>                                           (1 + 3) / 4]

+--------------------+--------+
|                Zone|count(1)|
+--------------------+--------+
|Eltingville/Annad...|       1|
|Governor's Island...|       1|
|       Arden Heights|       1|
|       Port Richmond|       3|
|       Rikers Island|       4|
|   Rossville/Woodrow|       4|
|         Great Kills|       4|
| Green-Wood Cemetery|       4|
|         Jamaica Bay|       5|
|         Westerleigh|      12|
|New Dorp/Midland ...|      14|
|       West Brighton|      14|
|             Oakwood|      14|
|        Crotona Park|      14|
|       Willets Point|      15|
|Breezy Point/Fort...|      16|
|Saint George/New ...|      17|
|       Broad Channel|      18|
|     Mariners Harbor|      21|
|Heartland Village...|      22|
+--------------------+--------+
only showing top 20 rows
